<div style="background: linear-gradient(120deg, #1ca9c9 0%, #40c4d6 60%, #7fe0e0 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Actividad 3: Extracción de archivos Parquet y JSON</h1>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
    </div>
</div>


# 1. De múltiples CSV a un dataset Parquet analítico

Usando los archivos de la carpeta `Data/csvs_ejemplo/`, el estudiante deberá:

1. **Consolidar** todos los CSV en un solo `DataFrame`.
2. **Limpiar** los datos (nulos, duplicados y columnas índice).
3. Añadir una columna `origen` con el nombre del archivo de procedencia.
4. Guardar como **Parquet particionado por `Ciudad`**.
5. Comparar **tamaño en disco** y **tiempo de lectura** entre CSV y Parquet.
6. Responder:
   - ¿Cuál es el promedio de `Números` por ciudad?
   - ¿Qué ciudad tiene más registros?
   - ¿Cuánto espacio ahorra Parquet frente a CSV?

In [19]:
# 1 Consolidar todos los CSV y
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

dir_base = os.getcwd()
print(dir_base)
ruta = os.path.join(dir_base, 'Data')
print(ruta)

ruta

os.listdir(ruta)

dfs = []
for archivo in os.listdir(carpeta_csv):
    if archivo.endswith('.csv'):
        # Leer archivo
        ruta_archivo = os.path.join(carpeta_csv, archivo)
        df = pd.read_csv(ruta_archivo)
        
        # Eliminar columnas índice si se generaron al guardar previamente (suelen llamarse 'Unnamed: 0')
        columnas_indice = [col for col in df.columns if 'Unnamed' in col]
        df = df.drop(columns=columnas_indice)
        
        # Añadir columna de origen
        df['origen'] = archivo
        dfs.append(df)

consolidado_df = pd.concat(dfs, ignore_index=True)

c:\Users\santi\OneDrive - ITESO\ITESO\Semestre 3\Procesamiento de Datos\Laboratorio_de_Procesamiento_de_Datos_02026\Modulo I
c:\Users\santi\OneDrive - ITESO\ITESO\Semestre 3\Procesamiento de Datos\Laboratorio_de_Procesamiento_de_Datos_02026\Modulo I\Data


In [15]:
# 2. Limpieza: quitar columnas índice, nulos y duplicados
consolidado_df = consolidado_df.drop_duplicates()
consolidado_df = consolidado_df.dropna()

In [17]:
# 4. Dataset Parquet particionado por Ciudad4
ruta_parquet = os.path.join(ruta, 'dataset_particionado.parquet')
ruta_csv_total = os.path.join(ruta, 'archivo_total.csv')

# Guardo en Parquet particionado
consolidado_df.to_parquet(ruta_parquet, partition_cols=['Ciudad'])

# Guardo un CSV consolidado para poder compararlos justamente
consolidado_df.to_csv(ruta_csv_total, index=False)

In [21]:
# 5 comparar tamaño y tiempo de lectura CSV vs Parquet.
# Hint: Para medir el tiempo de lectura, puedes usar la función time.time() antes y después de la lectura del archivo, y calcular la diferencia. Por ejemplo:
tamano_csv_mb = os.path.getsize(ruta_csv_total) / (1024 * 1024)

# Para Parquet particionado (al ser una carpeta, hay que sumar el tamaño de sus partes)
tamano_parquet_mb = 0
for raiz, directorios, archivos in os.walk(ruta_parquet):
    for f in archivos:
        tamano_parquet_mb += os.path.getsize(os.path.join(raiz, f))
tamano_parquet_mb = tamano_parquet_mb / (1024 * 1024)

# --- Comparación de tiempo ---
inicio_csv = time.time()
pd.read_csv(ruta_csv_total)
fin_csv = time.time()
tiempo_csv = fin_csv - inicio_csv

inicio_parquet = time.time()
pd.read_parquet(ruta_parquet)
fin_parquet = time.time()
tiempo_parquet = fin_parquet - inicio_parquet

print(f"Tamaño CSV: {tamano_csv_mb:.2f} MB  |  Tiempo lectura CSV: {tiempo_csv:.4f} s")
print(f"Tamaño Parquet: {tamano_parquet_mb:.2f} MB  |  Tiempo lectura Parquet: {tiempo_parquet:.4f} s")
#from time import time
#start_time = time.time()
# Código para leer el archivo
#end_time = time.time()
#print(f"Tiempo de lectura: {end_time - start_time} segundos")



Tamaño CSV: 0.00 MB  |  Tiempo lectura CSV: 0.0084 s
Tamaño Parquet: 0.05 MB  |  Tiempo lectura Parquet: 0.0629 s


In [22]:
# 6. Código y Respuestas de análisis
# ¿Cuál es el promedio de `Números` por ciudad?
promedio = consolidado_df.groupby('Ciudad')['Números'].mean()
print("\nPromedio de 'Números' por ciudad:")
print(promedio)

# ¿Qué ciudad tiene más registros?
ciudad_max = consolidado_df['Ciudad'].value_counts().idxmax()
print(f"\nCiudad con más registros: {ciudad_max}")

# ¿Cuánto espacio ahorra Parquet frente a CSV?
ahorro = tamano_csv_mb - tamano_parquet_mb
print(f"\nParquet ahorra {ahorro:.2f} MB de espacio frente al CSV consolidado.")


Promedio de 'Números' por ciudad:
Ciudad
Aguascalientes      16.00
Cancún              23.00
Ciudad de México    29.00
Guadalajara         25.75
Hermosillo          27.00
León                13.00
Monterrey           31.00
Morelia              9.00
Mérida              29.00
Oaxaca              30.00
Puebla              25.50
Querétaro            8.00
Saltillo            34.00
Tijuana             21.00
Toluca              14.00
Veracruz            18.00
Name: Números, dtype: float64

Ciudad con más registros: Guadalajara

Parquet ahorra -0.05 MB de espacio frente al CSV consolidado.



# 2. Extracción de archivos JSON

Utilizando el fichero `metallica.json` que se encuentra en la carpeta Data, que contiene información sobre los discos del grupo Metallica, crea distintas funciones en python, utilizando la librería `json`, que realicen las siguientes funciones:

1. **Listar información:** Listar el título del disco, y el contenido y fecha en el que se ha publicado en la wiki.
2. **Contar información:** Mostrar los títulos de discos y el número de canciones que tiene cada una.
3. **Buscar o filtrar información:** Recibe un límite inferior y otro superior, y muestra el título, y los nombres de las etiquetas ("tags") de los discos cuyos número de oyentes esté entre los dos límites.
4. **Buscar información relacionada:** Recibe el nombre una canción y muestra el título del disco donde está.

In [23]:
#1. Función paara lsitar el titulo del disco, y el contenido y fecha en el que se ha publicado en la wiki
import os
import json
import pandas as pd
from pandas import json_normalize

#carga inicial
json_metalica_path = os.path.join(ruta, "metallica.json")
with open(json_metalica_path, encoding='utf-8') as json_metalica:
    jsondata_metalica = json.load(json_metalica)

df_json_metalica = json_normalize(jsondata_metalica)

#listar información
def listar_informacion(df):
    """Devuelve el título del disco, contenido y fecha de publicación de la wiki."""
    return df[['album.name', 'album.wiki.published', 'album.wiki.content']]

In [ ]:
#2. Mostrar los títulos de discos y el número de canciones que tiene cada una.
def contar_canciones(df):
    """Devuelve los títulos de discos y el número de canciones que tiene cada uno."""
    # Usamos comprensión de listas para optimizar tu ciclo for
    num_canciones = [len(tracks) if isinstance(tracks, list) else 0 for tracks in df['album.tracks.track']]
    
    df_resultado = df[['album.name']].copy()
    df_resultado['num_canciones'] = num_canciones
    return df_resultado

In [25]:
#3. Recibe un límite inferior y otro superior, y muestra el título, y los nombres de las etiquetas ("tags") 
#   de los discos cuyos número de oyentes esté entre los dos límites.
def filtrar_por_oyentes(df, lim_inferior, lim_superior):
    """Filtra por rango de oyentes y muestra título y tags."""
    # Extraemos los tags de forma más directa
    lista_tags = []
    for tags in df['album.tags.tag']:
        nombres = [tg['name'] for tg in tags] if isinstance(tags, list) else []
        lista_tags.append(nombres)
        
    df_filtrado = df.copy()
    df_filtrado['nombres_tags'] = lista_tags
    df_filtrado['album.listeners'] = df_filtrado['album.listeners'].astype(int)
    
    # Aplicamos la máscara booleana
    mascara = (df_filtrado['album.listeners'] >= lim_inferior) & (df_filtrado['album.listeners'] <= lim_superior)
    
    return df_filtrado.loc[mascara, ['album.name', 'nombres_tags']]

In [26]:
# 4. Recibe el nombre una canción y muestra el título del disco donde está.
def buscar_disco_por_cancion(df, nombre_cancion):
    """Recibe una canción y muestra el título del disco donde está."""
    for index, row in df.iterrows():
        tracks = row['album.tracks.track']
        if isinstance(tracks, list):
            for track in tracks:
                # Comparamos ignorando mayúsculas/minúsculas para evitar errores
                if track.get('name', '').lower() == nombre_cancion.lower():
                    return row['album.name']
    
    return "Canción no encontrada"

In [27]:
# Probar Función 1
print(listar_informacion(df_json_metalica))

# Probar Función 2
print(contar_canciones(df_json_metalica))

# Probar Función 3
print(filtrar_por_oyentes(df_json_metalica, 1000, 500000))

# Probar Función 4
print(buscar_disco_por_cancion(df_json_metalica, "Master of Puppets"))

            album.name album.wiki.published  \
0         Kill 'Em All   01 Apr 2009, 22:18   
1   Ride the Lightning   26 Nov 2008, 18:21   
2    Master of Puppets   25 Jul 2008, 02:55   
3  And Justice For All   12 Jun 2012, 04:07   

                                  album.wiki.content  
0  Kill 'Em All is Metallica's first studio album...  
1  Ride The Lightning is the second album by thra...  
2  Master of Puppets is the third studio album by...  
3  …And Justice for All is the fourth studio albu...  
            album.name  num_canciones
0         Kill 'Em All             10
1   Ride the Lightning              8
2    Master of Puppets              8
3  And Justice For All              9
            album.name                                       nombres_tags
3  And Justice For All  [thrash metal, heavy metal, metal, albums I ow...
Master of Puppets
